# LiDAR Point Cloud Visualization with LidarOdometryWrapper

This notebook demonstrates how to use the `LidarOdometryWrapper` class to load a segment from pyboreas and visualize the LiDAR point cloud using plotly.

In [1]:
import os
import sys

module_path = os.path.abspath(os.path.join(".."))
if module_path not in sys.path:
    sys.path.append(module_path)

In [2]:
# Import necessary libraries
import plotly.graph_objects as go
import numpy as np
from src.dataset.utils import LidarOdometryWrapper

In [3]:
# Initialize the LidarOdometryWrapper
# This automatically loads the dataset and creates a rotation matrix around z-axis
wrapper = LidarOdometryWrapper()

In [4]:
START_INDEX = 5432
indices = [START_INDEX + i for i in range(5)]

# Use the wrapper's get_lidar_point_cloud method which handles rotation automatically
sample_points = []
for idx in indices:
    # This method returns already rotated points around z-axis
    sample_points.append(wrapper.get_lidar_point_cloud(idx))
    # Memory management is handled internally by the wrapper

In [5]:
from math import pi

MAX_DIST = 40.0

# Filter points by distance (same logic as before)
filtered_points = []
for points in sample_points:
    # Points are already rotated by the wrapper
    norms = np.linalg.norm(points, axis=1)  # Compute L2 norm for each point
    filtered_points.append(points[norms < MAX_DIST])

In [6]:
def draw_lidar_cloud(input_points: np.ndarray, n_points: int = 10000):
    points = input_points[
        np.random.choice(input_points.shape[0], n_points, replace=False)
    ]

    # Create a 3D scatter plot for the LiDAR point cloud
    fig = go.Figure(
        data=[
            go.Scatter3d(
                x=points[:, 0],
                y=points[:, 1],
                z=points[:, 2],
                mode="markers",
                marker=dict(
                    size=2,
                    color=points[:, 2],  # Color points by height
                    colorscale="Viridis",
                    opacity=0.7,
                ),
                name="LiDAR Points",
            )
        ]
    )

    # Update layout with custom styling
    fig.update_layout(
        title=dict(
            text="LiDAR Point Cloud Visualization", font=dict(size=20, family="Arial")
        ),
        scene=dict(
            xaxis_title="X (m)",
            yaxis_title="Y (m)",
            zaxis_title="Z (m)",
            camera=dict(eye=dict(x=1.5, y=1.5, z=1.0)),
            aspectmode="data",
        ),
        height=900,
        width=1200,
    )
    return fig

## Visualize one frame

In [7]:
fig = draw_lidar_cloud(filtered_points[0])
# Show the plot
fig.show()